# Memory Experiment - Hypergraph Product Code

Build and inspect three HGP instances through LightStim's public `MemoryExperiment` interface: `[[13,1,3]]` (unrotated surface), `[[18,2,3]]` (toric), and `[[225,9,4]]`. Batch sweeps use [`benchmarks/memory/run_memory.py`](../../benchmarks/memory/run_memory.py).

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.protocols import MemoryExperiment
from lightstim.qec_code.HGP import (
    HGPCodeExtractionBlock,
    hgp_13_1_3,
    hgp_18_2_3,
    hgp_225_9_4,
)

In [ ]:
BASIS = "Z"  # "Z" or "X"
SHOTS = 16


def build_memory(name, patch):
    experiment = MemoryExperiment(
        qec_patch=patch,
        patch_name=name,
        extraction_block_class=HGPCodeExtractionBlock,
        rounds=patch.code_distance,
        basis=BASIS,
        if_detector=True,
    )
    circuit = experiment.build().without_noise()
    sample = circuit.compile_detector_sampler().sample(
        SHOTS,
        append_observables=True,
    )
    assert not sample.any()
    print(
        f"{name}: [[{patch.num_data_qubits}, {patch.num_logicals}, "
        f"{patch.code_distance}]], total_qubits={circuit.num_qubits}, "
        f"detectors={circuit.num_detectors}, observables={circuit.num_observables}"
    )
    return circuit


patches = {
    "hgp_13_1_3": hgp_13_1_3(),
    "hgp_18_2_3": hgp_18_2_3(),
    "hgp_225_9_4": hgp_225_9_4(),
}
circuits = {
    name: build_memory(name, patch)
    for name, patch in patches.items()
}

## `[[13,1,3]]` - unrotated surface code

One complete bulk syndrome-extraction round (ticks 11-21), shown with `detslice-with-ops-svg`.

In [ ]:
circuits["hgp_13_1_3"].without_noise().diagram(
    "detslice-with-ops-svg", tick=range(11, 22)
)

## `[[18,2,3]]` - toric code

One complete bulk syndrome-extraction round (ticks 11-21), shown with `detslice-with-ops-svg`.

In [ ]:
circuits["hgp_18_2_3"].without_noise().diagram(
    "detslice-with-ops-svg", tick=range(11, 22)
)

## `[[225,9,4]]` - qLDPC instance

One complete bulk syndrome-extraction round (ticks 19-37), shown with `detslice-with-ops-svg`.

In [ ]:
circuits["hgp_225_9_4"].without_noise().diagram(
    "detslice-with-ops-svg", tick=range(19, 38)
)